In [3]:
from astroquery.mast import Catalogs, Observations
import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import lightkurve as lk
import pickle
from astropy.utils.data import clear_download_cache
from astropy.io import fits

In [4]:
data_folder = '/Users/philvanlane/Documents/lc_ae/data/'
lcs = pd.read_csv(data_folder + 'all_LC_prefix4_.csv')

### Download lightcurves and save them to file

In [42]:
len(lcs)

11228

In [31]:
all_lc_data = {}

In [5]:
with open(data_folder + "lc_data.pickle", "rb") as file:
    all_lc_data = pickle.load(file)

In [34]:
with open(data_folder + "lc_data_p4_.pickle", "rb") as file:
    all_lc_data = pickle.load(file)

In [6]:
all_lc_data.keys()

dict_keys(['400019820_51', '400066956_37', '400066956_38', '400066956_64', '400066956_65', '400098374_17', '400105331_19', '400105331_20', '400105331_26', '400105331_40', '400105331_47', '400125516_60', '400125516_73', '400166860_43', '400166860_44', '400166860_45', '400180890_11', '400184483_10', '400184483_11', '400320132_10', '400320132_11', '400320132_37', '400320132_38', '400361232_26', '400361232_52', '400361232_53', '40047077_3', '40047077_30', '40047077_70', '400513376_43', '400513376_44', '400513376_45', '400598180_8', '400598180_9', '400598180_35', '400598180_36', '400598180_62', '400598180_63', '400598180_89', '400598180_90', '400631549_11', '400631549_37', '400631549_38', '400631549_64', '400631549_65', '400721330_10', '400721330_11', '400721330_37', '400721330_38', '400721330_64', '400721330_65', '400891204_18', '400891204_19', '400891204_58', '400891204_59', '400891204_86', '400913139_10', '400913139_37', '400946473_52', '400953317_37', '401049758_10', '401049758_11', '40

In [44]:
for i in range(len(lcs)):
    if i % 500 == 0:
        print(f"Processing light curve {i}/{len(lcs)}")
        with open(data_folder + "lc_data.pickle", "wb") as file:
            pickle.dump(all_lc_data, file)
    tic_id = lcs['TIC_ID'][i]
    sector = lcs['sector'][i]
    key = f'{tic_id}_{sector}'
    if key in all_lc_data:
        continue
    res = lk.search_lightcurve(f"TIC {tic_id}", mission='TESS', sector=sector, author='SPOC', exptime=120)
    filt_res = res[(res.table['target_name'] == str(tic_id))]
    if len(filt_res) > 0:
        try:
            lc = filt_res[0].download()
            time = lc['time'].value
            flux = lc['pdcsap_flux'].value.filled(np.nan)
            flux_err = lc['pdcsap_flux_err'].value.filled(np.nan)

            # Metadata
            camera = lc.meta['CAMERA']
            ccd = lc.meta['CCD']


            # Mask out non-finite values
            mask = np.isfinite(flux) & np.isfinite(time)
            flux = flux[mask]
            time = time[mask]
            flux_err = flux_err[mask]

            # Normalize flux values
            flux_norm = (flux - np.mean(flux)) / np.std(flux)
            flux_err_norm = flux_err / np.std(flux)
            med_flux_err = np.nanmedian(flux_err_norm)
            mean_flux = np.mean(flux)
            std_flux = np.std(flux)

            # Saving to dict
            rec = {
                'TIC_ID': tic_id,
                'sector': sector,
                'time': time,
                'flux': flux_norm,
                'flux_err': flux_err_norm,
                'metadata': {
                    'TIC_ID': tic_id,
                    'sector': sector,
                    'camera': camera,
                    'ccd': ccd,
                    'mean_flux': mean_flux,
                    'std_flux': std_flux
                }
            }
            all_lc_data[key] = rec
        except Exception as e:
            with open("log.txt", "a") as f:
                f.write(f"⚠️ Download failed for TIC {tic_id}, sector {sector}: {e}\n")
    else:
        with open("log.txt", "a") as f:
            f.write(f"⚠️ No light curve data found for TIC {tic_id} in sector {sector}\n")
    clear_download_cache()

Processing light curve 0/11228
Processing light curve 500/11228
Processing light curve 1000/11228
Processing light curve 1500/11228
Processing light curve 2000/11228
Processing light curve 2500/11228
Processing light curve 3000/11228
Processing light curve 3500/11228
Processing light curve 4000/11228


Processing light curve 4500/11228
Processing light curve 5000/11228
Processing light curve 5500/11228
Processing light curve 6000/11228
Processing light curve 6500/11228
Processing light curve 7000/11228
Processing light curve 7500/11228
Processing light curve 8000/11228
Processing light curve 8500/11228
Processing light curve 9000/11228
Processing light curve 9500/11228
Processing light curve 10000/11228
Processing light curve 10500/11228
Processing light curve 11000/11228


In [86]:
tic_id = 449348769
sector = 37
key = f'{tic_id}_{sector}'

In [87]:
res = lk.search_lightcurve(f"TIC {tic_id}", mission='TESS', sector=sector, author='SPOC', exptime=120)

In [88]:
res

#,mission,year,author,exptime,target_name,distance
,,,,s,,arcsec
0,TESS Sector 37,2021,SPOC,120,449348769,0.0


In [89]:
lc = res[0].download()
time = lc['time'].value
flux = lc['pdcsap_flux'].value.filled(np.nan)
flux_err = lc['pdcsap_flux_err'].value.filled(np.nan)

# Metadata
camera = lc.meta['CAMERA']
ccd = lc.meta['CCD']


# Mask out non-finite values
mask = np.isfinite(flux) & np.isfinite(time)
flux = flux[mask]
time = time[mask]
flux_err = flux_err[mask]

# Normalize flux values
flux_norm = (flux - np.mean(flux)) / np.std(flux)
flux_err_norm = flux_err / np.std(flux)
med_flux_err = np.nanmedian(flux_err_norm)
mean_flux = np.mean(flux)
std_flux = np.std(flux)

# Saving to dict
rec = {
    'TIC_ID': tic_id,
    'sector': sector,
    'time': time,
    'flux': flux_norm,
    'flux_err': flux_err_norm,
    'metadata': {
        'TIC_ID': tic_id,
        'sector': sector,
        'camera': camera,
        'ccd': ccd,
        'mean_flux': mean_flux,
        'std_flux': std_flux
    }
}

In [90]:
key

'449348769_37'

In [91]:
rec

{'TIC_ID': 449348769,
 'sector': 37,
 'time': array([2308.13465847, 2308.13604739, 2308.13743631, ..., 2332.58063025,
        2332.58201912, 2332.58340799]),
 'flux': array([ 0.14380282, -0.39976805, -0.0745539 , ...,  1.1854733 ,
         1.7463801 , -0.548118  ], dtype=float32),
 'flux_err': array([0.9360117 , 0.93584484, 0.9356327 , ..., 0.8531779 , 0.8537988 ,
        0.8524078 ], dtype=float32),
 'metadata': {'TIC_ID': 449348769,
  'sector': 37,
  'camera': 2,
  'ccd': 3,
  'mean_flux': 18276.604,
  'std_flux': 20.617424}}

In [92]:
all_lc_data[key] = rec

In [27]:
clear_download_cache()

In [93]:
len(all_lc_data)

11228

In [94]:
with open(data_folder + "lc_data_p4_.pickle", "wb") as file:
    pickle.dump(all_lc_data, file)

In [ ]:
lc

time,flux,flux_err,timecorr,cadenceno,centroid_col,centroid_row,sap_flux,sap_flux_err,sap_bkg,sap_bkg_err,pdcsap_flux,pdcsap_flux_err,quality,psf_centr1,psf_centr1_err,psf_centr2,psf_centr2_err,mom_centr1,mom_centr1_err,mom_centr2,mom_centr2_err,pos_corr1,pos_corr2,time_adj
,electron / s,electron / s,d,,pix,pix,electron / s,electron / s,electron / s,electron / s,electron / s,electron / s,,pix,pix,pix,pix,pix,pix,pix,pix,pix,pix,
Time,float32,float32,float32,int32,float64,float64,float32,float32,float32,float32,float32,float32,int32,float64,float32,float64,float32,float64,float32,float64,float32,float32,float32,TimeDelta
1624.9610577298067,———,———,5.7410472e-03,286200,1289.24070,68.15741,1.6240644e+05,5.8845181e+01,6.5234281e+04,3.0011850e+01,———,———,11000000000000,———,———,———,———,1289.24070,3.2716387e-04,68.15741,3.6591874e-04,———,———,0.0
1624.9624466246237,———,———,5.7410537e-03,286201,1289.24228,68.15370,1.6246848e+05,5.8917679e+01,6.5533492e+04,3.0090111e+01,———,———,11000000000000,———,———,———,———,1289.24228,3.2758922e-04,68.15370,3.6630200e-04,———,———,0.001388894816955144
1624.9638355194415,———,———,5.7410602e-03,286202,1289.24444,68.15037,1.6250508e+05,5.9004112e+01,6.5910789e+04,3.0178328e+01,———,———,11000000000000,———,———,———,———,1289.24444,3.2797770e-04,68.15037,3.6676010e-04,———,———,0.002777789634819783
1624.9652244137933,———,———,5.7410663e-03,286203,1289.24690,68.14477,1.6252361e+05,5.9096687e+01,6.6355148e+04,3.0272585e+01,———,———,11000000000000,———,———,———,———,1289.24690,3.2838774e-04,68.14477,3.6741034e-04,———,———,0.004166683986568387
1624.9666133086112,———,———,5.7410728e-03,286204,1289.23683,68.16043,1.6244284e+05,5.9144810e+01,6.6629391e+04,3.0328495e+01,———,———,11000000000000,———,———,———,———,1289.23683,3.2890984e-04,68.16043,3.6793906e-04,———,———,0.005555578804433026
1624.9680022034286,———,———,5.7410793e-03,286205,1289.24638,68.14424,1.6258173e+05,5.9245548e+01,6.7062367e+04,3.0422840e+01,———,———,11000000000000,———,———,———,———,1289.24638,3.2922227e-04,68.14424,3.6838220e-04,———,———,0.006944473621842917
1624.9693910982464,———,———,5.7410859e-03,286206,1289.24028,68.15482,1.6252508e+05,5.9297226e+01,6.7289836e+04,3.0488157e+01,———,———,11000000000000,———,———,———,———,1289.24028,3.2956645e-04,68.15482,3.6896588e-04,———,———,0.008333368439707556
1624.9707799930638,———,———,5.7410924e-03,286207,1289.24277,68.14882,1.6248873e+05,5.9367722e+01,6.7673750e+04,3.0560446e+01,———,———,11000000000000,———,———,———,———,1289.24277,3.3015019e-04,68.14882,3.6937551e-04,———,———,0.009722263257117447
